# 02 — Feature Extraction

Frame bazında RMS, RMS seviyesi ve spectral centroid hesaplama. Örnekleme hızı korunur; analiz için kanalların ortalaması alınır.

## Bağımlılıklar ve dosya yolları

`AUDIO_PATH = None`, üç kısa darbe içeren sentetik örneği oluşturur. Mevcut bir WAV için dosya yolu kullanılır. `FRAME_LENGTH`, her frame'in örnek sayısı; `HOP_LENGTH`, frame merkezleri arasındaki örnek sayısıdır.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import importlib.metadata
import json
import platform
import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from IPython.display import Audio, display

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
AUDIO_PATH = None  # Örnek: ROOT / "data" / "raw" / "impact.wav"
FRAME_LENGTH = 1024
HOP_LENGTH = 256
ACTIVE_THRESHOLD_DB = -60.0
FIGURES = ROOT / "results" / "figures" / "02_feature_extraction"
TABLES = ROOT / "results" / "tables" / "02_feature_extraction"
EXPERIMENT = ROOT / "experiments" / "02_feature_extraction"
for folder in [FIGURES, TABLES, EXPERIMENT]:
    folder.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (11, 3), "axes.grid": False})


## WAV okuma

Sentetik sinyalde onset değerleri 0.4, 1.2 ve 2.0 saniyedir. Her darbe sinüs bileşenleri, gürültü ve üstel sönme zarfı içerir. Seed 42, gürültü üretimini tekrarlanabilir kılar. PCM_16 yazımı sonrası dosya tekrar okunur.

`always_2d=True` çıktıyı `(örnek, kanal)` biçiminde tutar. Kanal ortalaması stereo bilgiyi kaybettirir; karşı fazlı kanallar birbirini söndürebilir.

In [ ]:
using_demo = AUDIO_PATH is None
if using_demo:
    demo_sr = 24000
    t = np.arange(3 * demo_sr) / demo_sr
    demo = np.zeros_like(t)
    rng = np.random.default_rng(42)
    for onset in [0.4, 1.2, 2.0]:
        tau = t - onset
        active = tau >= 0
        u = tau[active]
        demo[active] += np.exp(-u / 0.12) * (
            0.6 * np.sin(2 * np.pi * 220 * u)
            + 0.25 * np.sin(2 * np.pi * 1400 * u)
            + 0.15 * rng.standard_normal(len(u)))
    demo *= 0.8 / max(np.max(np.abs(demo)), 1e-12)
    AUDIO_PATH = ROOT / "data" / "generated" / "audio_basics_demo.wav"
    AUDIO_PATH.parent.mkdir(parents=True, exist_ok=True)
    sf.write(AUDIO_PATH, demo, demo_sr, subtype="PCM_16")
else:
    AUDIO_PATH = Path(AUDIO_PATH).expanduser()
    if not AUDIO_PATH.is_absolute():
        AUDIO_PATH = ROOT / AUDIO_PATH
if not AUDIO_PATH.is_file():
    raise FileNotFoundError(f"WAV bulunamadı: {AUDIO_PATH}")
audio, sr = sf.read(AUDIO_PATH, dtype="float64", always_2d=True)
if len(audio) == 0 or not np.isfinite(audio).all():
    raise ValueError("Dosya boş veya geçersiz örnek içeriyor.")
y = audio.mean(axis=1)
duration = len(y) / sr
print(f"Örnekleme hızı: {sr} Hz | Süre: {duration:.3f} s | Kanal: {audio.shape[1]}")
print(f"Frame: {1000*FRAME_LENGTH/sr:.2f} ms | Hop: {1000*HOP_LENGTH/sr:.2f} ms")
if np.max(np.abs(y)) <= 1:
    display(Audio(y, rate=sr, normalize=False))
else:
    print("Mono tepe genliği ±1 sınırını aşıyor; oynatma atlandı.")


## Frame bazında RMS

$\mathrm{RMS}[m]=\sqrt{\frac{1}{L}\sum_{n=0}^{L-1}y_m[n]^2}$

`L` frame uzunluğu, `m` frame indeksidir. Tüm kayıt için tek sayı yerine her kısa parçada ayrı bir seviye ölçülür. `center=True` ile frame merkezi `m * hop_length / sr` zamanına karşılık gelir; kenarlara sıfır eklenir.

RMS burada waveform üzerinden dikdörtgen pencerede hesaplanır. Bu hesap A-weighting uygulamaz ve algısal loudness ölçümü değildir.

RMS seviyesi $20\log_{10}(\mathrm{RMS}/1)$ ile hesaplanır; referans dijital genlik 1'dir (dBFS RMS). Bu tanımda genliği 1 olan sinüsün RMS seviyesi yaklaşık −3.01 dB'dir. SPL veya LUFS değildir. Sayısal gösterimde sıfır genlik için $10^{-10}$ alt sınırı kullanılır.

In [ ]:
rms = librosa.feature.rms(
    y=y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH,
    center=True, pad_mode="constant", dtype=np.float64)[0]
frame_times = np.arange(len(rms)) * HOP_LENGTH / sr
rms_db = 20 * np.log10(np.maximum(rms, 1e-10))
active_frames = rms_db > ACTIVE_THRESHOLD_DB
print(f"Frame sayısı: {len(rms)} | Eşik üstü frame: {active_frames.sum()}")


## Spectral centroid

$C[m]=\frac{\sum_k f_k |X[k,m]|}{\sum_k |X[k,m]|}$

`f_k` frekans kutusunun Hz değeridir. Büyüklük spectrogramı ağırlık olarak kullanılır; güç spectrogramı kullanımı farklı sonuç verir. Centroid, frekans içeriğinin ağırlık merkezidir ve brightness için bir göstergedir; pitch değildir.

STFT, RMS ile aynı frame uzunluğu, hop ve merkezleme kullanır. STFT'de Hann pencere uygulanır; waveform RMS hesabında uygulanmaz. Zaman noktaları eşleşir, fakat iki ölçümün pencere ağırlıkları farklıdır.

`ACTIVE_THRESHOLD_DB = -60`, bu deneyde düşük seviyeli frame'leri centroid grafiği ve özetinden çıkarmak için seçilen eşiktir. Bir onset dedektörü veya makalenin değerlendirme protokolü değildir. Sessiz/düşük seviyeli frame'lerin centroid değeri `NaN` olarak işaretlenir.

In [ ]:
D = librosa.stft(y=y, n_fft=FRAME_LENGTH, win_length=FRAME_LENGTH,
                 hop_length=HOP_LENGTH, window="hann", center=True,
                 pad_mode="constant")
magnitude = np.abs(D)
centroid_raw = librosa.feature.spectral_centroid(S=magnitude, sr=sr)[0]
centroid_hz = np.where(active_frames, centroid_raw, np.nan)
assert len(rms) == len(centroid_raw) == D.shape[1]
assert np.isfinite(rms).all() and np.isfinite(centroid_raw).all()
print("Centroid özeti: yalnızca eşik üstü frame'ler")
if active_frames.any():
    print(f"Ortalama: {np.mean(centroid_hz[active_frames]):.2f} Hz")
else:
    print("Eşik üstü frame yok; centroid özeti tanımsız.")


## Özellik eğrileri

Üç panel waveform, frame RMS seviyesi ve spectral centroid'i aynı zaman ekseninde gösterir. Kesikli yatay çizgi RMS eşiğidir. Centroid eğrisindeki boşluklar eşik altındaki frame'lerdir.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
# Grafik boyutunu sınırlamak için waveform en fazla yaklaşık 3000 noktada çizilir.
stride = max(1, int(np.ceil(len(y) / 3000)))
sample_times = np.arange(len(y)) / sr
axes[0].plot(sample_times[::stride], y[::stride], linewidth=0.7)
axes[0].set(ylabel="Genlik", title="Waveform")
axes[1].plot(frame_times, rms_db, linewidth=1)
axes[1].axhline(ACTIVE_THRESHOLD_DB, color="gray", linestyle="--", label="Analiz eşiği")
axes[1].set(ylabel="RMS (dBFS, ref=1)", title="Frame RMS")
axes[1].legend()
axes[2].plot(frame_times, centroid_hz, linewidth=1)
axes[2].set(ylabel="Centroid (Hz)", xlabel="Zaman (s)", title="Spectral centroid")
axes[2].set_xlim(0, duration)
plt.tight_layout()
fig.savefig(FIGURES / "feature_curves.svg", bbox_inches="tight")
plt.show()


## Genlik ölçekleme karşılaştırması

`0.5 * y` üzerinde özellikler tekrar hesaplanır. Pozitif sabit ölçekleme RMS'i aynı oranda değiştirir. Centroid formülündeki pay ve payda aynı katsayıyla çarpıldığı için centroid değişmez.

Karşılaştırmada her iki sinyal için özgün sinyalin eşik üstü frame maskesi kullanılır. Böylece ölçekleme sonrası eşik değişiminin etkisi ölçüme karışmaz. Bu kontrol, model controllability veya disentanglement testi değildir.

In [ ]:
quieter = 0.5 * y
rms_half = librosa.feature.rms(
    y=quieter, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH,
    center=True, pad_mode="constant", dtype=np.float64)[0]
D_half = librosa.stft(y=quieter, n_fft=FRAME_LENGTH, win_length=FRAME_LENGTH,
                      hop_length=HOP_LENGTH, window="hann", center=True,
                      pad_mode="constant")
centroid_half = librosa.feature.spectral_centroid(S=np.abs(D_half), sr=sr)[0]
if active_frames.any():
    rms_ratios = rms_half[active_frames] / rms[active_frames]
    level_difference_db = 20 * np.log10(rms_ratios)
    centroid_difference_hz = np.abs(centroid_half[active_frames] - centroid_raw[active_frames])
    mean_level_difference_db = float(np.mean(level_difference_db))
    max_centroid_difference_hz = float(np.max(centroid_difference_hz))
    print(f"Ortalama RMS oranı: {np.mean(rms_ratios):.6f}")
    print(f"Ortalama RMS seviye farkı: {mean_level_difference_db:.6f} dB")
    print(f"Maksimum centroid farkı: {max_centroid_difference_hz:.9f} Hz")
    assert np.allclose(rms_ratios, 0.5, atol=1e-10)
    assert np.allclose(centroid_half[active_frames], centroid_raw[active_frames], atol=1e-6)
else:
    mean_level_difference_db = None
    max_centroid_difference_hz = None
    print("Eşik üstü frame yok; karşılaştırma atlandı.")


## Çıktı dosyaları

`frame_features.csv`, her frame'in zamanı, RMS değeri, RMS seviyesi ve maskelenmiş centroid'ini içerir. Tanımsız centroid alanları boş kaydedilir. `summary.csv` özet ölçümleri, `run.json` giriş hash'i, ayarları ve ortam sürümlerini tutar. Aynı çıktı yolları sonraki çalıştırmada güncellenir.

In [ ]:
with (TABLES / "frame_features.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["time_s", "rms", "rms_dbfs_ref_1", "centroid_hz", "active_frame"])
    for time, level, db, centroid, active in zip(frame_times, rms, rms_db, centroid_hz, active_frames):
        writer.writerow([float(time), float(level), float(db),
                         float(centroid) if active else "", int(active)])
metrics = {
    "sample_rate_hz": int(sr), "duration_s": duration,
    "channels": int(audio.shape[1]), "frames": len(rms),
    "active_frames": int(active_frames.sum()),
    "frame_ms": 1000*FRAME_LENGTH/sr, "hop_ms": 1000*HOP_LENGTH/sr,
    "mean_rms_all_frames": float(np.mean(rms)),
    "max_frame_rms": float(np.max(rms)),
    "mean_centroid_active_hz": float(np.mean(centroid_hz[active_frames])) if active_frames.any() else None,
    "half_amplitude_mean_delta_db": mean_level_difference_db,
    "half_amplitude_max_centroid_difference_hz": max_centroid_difference_hz,
}
with (TABLES / "summary.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerows(metrics.items())
try:
    source_path = str(AUDIO_PATH.resolve().relative_to(ROOT.resolve()))
except ValueError:
    source_path = AUDIO_PATH.name
run = {
    "experiment_id": "02_feature_extraction",
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "source": source_path, "synthetic_demo": using_demo,
    "audio_sha256": hashlib.sha256(AUDIO_PATH.read_bytes()).hexdigest(),
    "python": platform.python_version(), "platform": platform.system(),
    "packages": {p: importlib.metadata.version(p) for p in
                 ["numpy", "librosa", "soundfile", "matplotlib"]},
    "analysis": {"frame_length": FRAME_LENGTH, "hop_length": HOP_LENGTH,
                 "rms_window": "rectangular", "stft_window": "hann",
                 "center": True, "pad_mode": "constant",
                 "rms_db_reference": 1, "rms_floor": 1e-10,
                 "active_threshold_db": ACTIVE_THRESHOLD_DB,
                 "centroid_weights": "STFT magnitude", "comparison_mask": "original active frames"},
    "metrics": metrics,
    "scope": "unweighted frame RMS and centroid; not Sketch2Sound reproduction",
}
(EXPERIMENT / "run.json").write_text(
    json.dumps(run, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
print("Kaydedildi:", FIGURES.relative_to(ROOT), TABLES.relative_to(ROOT),
      (EXPERIMENT / "run.json").relative_to(ROOT))
